# D-Fire — Phase 1: Dataset Exploration

Validates dataset integrity, computes statistics, and visualizes samples.

**Class convention (verified):** `0 = smoke`, `1 = fire`.

Runs locally or on Kaggle. On Kaggle, attach the D-Fire YOLO dataset and set
`DATA_ROOT` to its mount path (e.g. `/kaggle/input/smoke-fire-detection-yolo`).

In [ ]:
import sys, os
from pathlib import Path

# Make the repo importable both locally and on Kaggle.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

# Set this to your dataset root. Leave as None to auto-detect (./data or /kaggle/input/*).
DATA_ROOT = None  # e.g. "/kaggle/input/smoke-fire-detection-yolo"
if DATA_ROOT:
    os.environ["DFIRE_ROOT"] = DATA_ROOT

In [ ]:
# 1) Validate dataset integrity (labels, coordinates, negatives, orphans)
from src.data.validate_dataset import validate_dataset
from src.utils import resolve_data_root

data_root = resolve_data_root(DATA_ROOT)
report = validate_dataset(data_root, check_images=False)
report["totals"]

In [ ]:
# 2) Compute statistics + figures + reports/dataset_report.md
from src.data.analyze_dataset import analyze_split, _aggregate_totals
from src.data.dfire import SPLITS

per_split = {s: analyze_split(data_root, s) for s in SPLITS}
totals = _aggregate_totals(per_split)
print("Total images:", totals["n_images"])
print("Boxes per class:", totals["class_box_counts"])
print("Categories:", totals["categories"])
totals["area_buckets"]

In [ ]:
# 3) Visualize a few annotated samples (ground-truth boxes)
import matplotlib.pyplot as plt
from src.data.dfire import iter_split, parse_label_file, yolo_to_pixel, CLASS_NAMES
from src.visualization.plot_predictions import draw_boxes, _read_rgb

samples = []
for img_path, lbl_path in iter_split(data_root / "train" / "images"):
    boxes, _ = parse_label_file(lbl_path)
    if boxes:
        img = _read_rgb(img_path)
        h, w = img.shape[:2]
        px = [(b.cls, *yolo_to_pixel(b, w, h), None) for b in boxes]
        samples.append((draw_boxes(img, px), f"{img_path.name} ({len(boxes)} boxes)"))
    if len(samples) >= 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (im, cap) in zip(axes.ravel(), samples):
    ax.imshow(im); ax.set_title(cap, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()